In [ ]:
import os
os.chdir("..")

In [ ]:
import logging, pprint
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import SGD, Adam
from scripts.pytorch_nn.supervised_trainer import PytorchSupervisedTrainer
from scripts.pytorch_nn.pointnet import PointNetRegressor, PointNetBasic

In [ ]:
logging.basicConfig(level=logging.INFO)

# Data

In [ ]:
class SyntheticLidarData:
    MAX_X = 10**2
    MAX_Y = 10**2
    MAX_Z = 10**2
    POINTS_PER_OBJECT = 10**4
    CLASS_0_SUBSET = 100
    CLASS_1_SUBSET = 100

    def generate_synthetic_data(self, n: int) -> dict[int, np.ndarray]:
        x = np.array([self._generate_cloud() for i in range(n)] + [self._generate_pole() for i in range(n)])
        y = np.array([0 for i in range(n)] + [0 for i in range(n)])
        return {"x": x, "y": y}

    def _generate_cloud(self) -> np.ndarray:
        random_x = [x + np.random.rand() for x in np.random.randint(0, self.MAX_X+1, self.POINTS_PER_OBJECT)]
        random_y = [x + np.random.rand() for x in np.random.randint(0, self.MAX_Y+1, self.POINTS_PER_OBJECT)]
        random_z = [x + np.random.rand() for x in np.random.randint(0, self.MAX_Z+1, self.POINTS_PER_OBJECT)]
        points = [np.array([x, y, z]) for x, y, z in zip(random_x, random_y, random_z)]
        return points

    def _generate_pole(self) -> np.ndarray:
        random_x = [x + np.random.rand() for x in np.random.randint(0, 11, self.POINTS_PER_OBJECT)]
        random_y = [x + np.random.rand() for x in np.random.randint(0, self.MAX_Y+1, self.POINTS_PER_OBJECT)]
        random_z = [x + np.random.rand() for x in np.random.randint(0, 11, self.POINTS_PER_OBJECT)]
        points = [np.array([x, y, z]) for x, y, z in zip(random_x, random_y, random_z)]
        return points

In [ ]:
class CustomDataset(Dataset):

    def __init__(self, raw_data: dict[int, np.ndarray]):
        self.x = raw_data["x"]
        self.y = raw_data["y"]
        print()
        print(f"X shape = {self.x.shape}")
        print(f"Y shape = {self.y.shape}")
        print()
        assert len(self.x) == len(self.y)

    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, index):
        return self.x[index].astype(np.float32), self.y[index]

In [ ]:
raw_train_data = SyntheticLidarData().generate_synthetic_data(100)
raw_test_data = SyntheticLidarData().generate_synthetic_data(100)

## Example train objects

In [ ]:
points = raw_train_data["x"][0]
y = 0

fig = go.Figure()
fig.add_trace(
    go.Scatter3d(
        x=[x[0] for x in points],
        y=[x[1] for x in points],
        z=[x[2] for x in points],
        marker=dict(size=1),
        mode="markers"
    )
)
fig.update_layout(
    title=f"Object with ground truth (y) {y}",
    width=600,
    scene=dict(
        xaxis=dict(range=(0, SyntheticLidarData.MAX_X)),
        yaxis=dict(range=(0, SyntheticLidarData.MAX_Y)),
        zaxis=dict(range=(0, SyntheticLidarData.MAX_Z))
    )
)
fig.show()


In [ ]:
points = raw_train_data["x"][190]
y = 1

fig = go.Figure()
fig.add_trace(
    go.Scatter3d(
        x=[x[0] for x in points],
        y=[x[1] for x in points],
        z=[x[2] for x in points],
        marker=dict(size=1),
        mode="markers"
    )
)
fig.update_layout(
    title=f"Object with ground truth (y) {y}",
    width=600,
    scene=dict(
        xaxis=dict(range=(0, SyntheticLidarData.MAX_X)),
        yaxis=dict(range=(0, SyntheticLidarData.MAX_Y)),
        zaxis=dict(range=(0, SyntheticLidarData.MAX_Z))
    )
)
fig.show()

# Training Parameters

In [ ]:
LEARNING_RATE = 1e-5
BATCH_SIZE = 25
EPOCHS = 2

# Training

In [ ]:
model = PointNetRegressor(pointnet=PointNetBasic())
loss_fn = nn.MSELoss()
optimizer = SGD(model.parameters(), lr=LEARNING_RATE)
train_dataloader = DataLoader(CustomDataset(raw_train_data), batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(CustomDataset(raw_test_data), batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
model, history = PytorchSupervisedTrainer().train(
    n_epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    model=model,
    loss_fn=loss_fn,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    optimizer=optimizer
)

In [ ]:
fig = go.Figure()


x, train, test = [], [], []
for epoch, data in history.items():
    x += [epoch+1 for i in range(len(data["train"]))]
    train += data["train"]
    test += [data["test"][0] for i in range(len(data["train"]))]

fig.add_trace(
    go.Scatter(
        name="Train loss",
        x=x,
        y=train,
        mode="lines"
    )
)
fig.add_trace(
    go.Scatter(
        name="Test loss",
        x=x,
        y=test,
        mode="lines"
    )
)

fig.update_layout(
    title="Training History",
    xaxis=dict(
        range=(0.75, epoch+1.25),
        title="Epoch",
        tickvals=[i+1 for i in range(epoch+1)]
        # type='category'
    ),
    yaxis=dict(title="Loss"),
    width=1000
)
fig.show()

In [ ]:
pprint.pp(history)